# I will put underscores inbetween different words within the column names

## I make above transformation for a single file (Testing only)

In [0]:
# list all directories in Silver container in datalake
dbutils.fs.ls('abfss://silver@intechstorage.dfs.core.windows.net/SalesLT/')

In [0]:
# List all files/folders in Gold container
dbutils.fs.ls('abfss://gold@intechstorage.dfs.core.windows.net/SalesLT/')

In [0]:
# Load single file called Address (note that the file is in Delta format because I converted the parquet files in Bronze container to Delta in the Silver container)
df = spark.read.format('delta').load('abfss://silver@intechstorage.dfs.core.windows.net/SalesLT/Address')

# commented out as it is not required for Data factory job
# display(df)

In [0]:
from pyspark.sql.functions import col

In [0]:
def rename_columns_to_snake_case(df):
    """
    Convert column names from PascalCase or camelCase to snake_case in a PySpark DataFrame.
    
    Args:
        df (DataFrame): The input DataFrame with columns to be renamed.
        
    Returns:
        DataFrame: A new DataFrame with column names converted to snake_case.
    """
    # Get the list of column names
    column_names = df.columns
    
    # Dictionary to hold old and new column name mappings
    rename_map = {}
    
    for old_col_name in column_names:
        # Convert column name from PascalCase or camelCase to snake_case
        new_col_name = "".join([
            "_" + char.lower() if (
                char.isupper()
                and idx > 0
                and not old_col_name[idx - 1].isupper()
            ) else char.lower()
            for idx, char in enumerate(old_col_name)
        ]).lstrip("_")
        
        # Avoid renaming to an existing column name
        if new_col_name in rename_map.values():
            raise ValueError(f"Duplicate column name found after renaming: '{new_col_name}'")
            
        # Map the old column name to the new column name
        rename_map[old_col_name] = new_col_name
        
    # Rename columns using the mapping
    for old_col_name, new_col_name in rename_map.items():
        df = df.withColumnRenamed(old_col_name, new_col_name)
        
    return df


In [0]:
# Display dataframe with return after function
new_df = rename_columns_to_snake_case(df)

# commented out as it is not required for Data factory job
# display(new_df)

## Doing the above transformation on all tables in all files and outputting the files in Gold container

This is done by running the function **rename_columns_to_snake_case()** via a for loop

In [0]:
# Initialise an empty list for table namce
table_name = []

# Append delta table names in Silver container to empty list
for i in dbutils.fs.ls('abfss://silver@intechstorage.dfs.core.windows.net/SalesLT/'):
    table_name.append(i.name.split('/')[0])

# Loop through each table to clean the column structures
for i in table_name:
    # Read each item from Silver container
    input_path = f'abfss://silver@intechstorage.dfs.core.windows.net/SalesLT/{i}/'
    df = spark.read.format('delta').load(input_path)
    
    # Rename all column headings to snake_case using function rename_columns_to_snake_case
    df = rename_columns_to_snake_case(df)
    
    # Output files in Delta format inside Gold container
    output_path = f'abfss://gold@intechstorage.dfs.core.windows.net/SalesLT/{i}/'
    df.write.format('delta').mode('overwrite').save(output_path)

print("Silver-to-Gold transformation completed! All table headers are now natively in snake_case.")
    